# Stage III K=1 vs K=80 regret comparison

This notebook evaluates the saved Stage III models with the finaltest-style regret search, comparing K=1 and K=80 on the exact same valuation profiles.

For each model/repeat, the valuation profiles and the grid-search best reports are shared. The optional initialization pool is generated once with `MAX_K = 80`; K=1 uses the first candidate from each optional pool, while K=80 uses all 80.

The three optional initialization counts are compared through `K_VALUES = [1, 80]`:

- `num_global_random = K`
- `num_local_around_truth = K`
- `num_local_around_best = K`


In [1]:
from pathlib import Path
import random
import time

import numpy as np
import torch
import torch.nn.functional as F

import networks as networks_mod
import utils as utils_mod
from networks import MixedWrapper
from restrictedAdam import Adam
from utils import utility, misreportUtility, loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(5)
networks_mod.device = device
utils_mod.device = device

print("device:", device)


device: cuda


In [2]:
MODEL_SPECS = [
    {"name": "2x2",  "path": Path("22stage3.pth"), "nAgent": 2, "nObject": 2},
    {"name": "2x5",  "path": Path("25stage3.pt"),  "nAgent": 2, "nObject": 5},
    {"name": "3x10", "path": Path("310stage3.pt"), "nAgent": 3, "nObject": 10},
    {"name": "5x10", "path": Path("510stage3.pt"), "nAgent": 5, "nObject": 10},
]

# Main test settings. K_VALUES are compared on the exact same valuation profiles.
K_VALUES = [1, 80]
MAX_K = max(K_VALUES)
N_BATCH = 1000
R = 300
GAMMA = 0.001
MINIMUM = 0.0
MAXIMUM = 1.0
GRID_SIZE = 1000
N_REPEATS = 1
SEED = 20260804

# Chunking keeps k=80 manageable on smaller GPUs. Set INIT_CHUNK_SIZE = None for fastest full-batch optimization.
GRID_CHUNK_SIZE = 100
INIT_CHUNK_SIZE = 32
# This matches the existing finaltest behavior: if a saved model is a MixedWrapper, evaluate mechanism.base.
USE_BASE_MODEL = True

missing = [str(spec["path"]) for spec in MODEL_SPECS if not spec["path"].exists()]
if missing:
    raise FileNotFoundError("Missing model file(s): " + ", ".join(missing))


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def torch_load_full_model(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def move_model_to_device(model):
    model = model.to(device)
    if hasattr(model, "device"):
        model.device = device
    if hasattr(model, "base") and hasattr(model.base, "device"):
        model.base.device = device
    return model


def freeze_parameters(model):
    states = [(param, param.requires_grad) for param in model.parameters()]
    for param, _ in states:
        param.requires_grad_(False)
    return states


def restore_parameters(states):
    for param, requires_grad in states:
        param.requires_grad_(requires_grad)


def get_eval_mechanism(model, use_base_model=True):
    model = move_model_to_device(model)
    if use_base_model and hasattr(model, "base"):
        eval_model = model.base
    else:
        eval_model = model
        if hasattr(eval_model, "st"):
            eval_model.st = False
    eval_model = move_model_to_device(eval_model)
    eval_model.eval()
    return eval_model


def forward_in_batches(mechanism, values, forward_batch_size=None):
    if forward_batch_size is None or values.shape[0] <= forward_batch_size:
        return mechanism(values)

    allocations = []
    payments = []
    for start in range(0, values.shape[0], forward_batch_size):
        end = min(start + forward_batch_size, values.shape[0])
        alloc, pay = mechanism(values[start:end])
        allocations.append(alloc)
        payments.append(pay)
    return torch.cat(allocations, dim=0), torch.cat(payments, dim=0)


def grid_best_single_item_misreports(
    eval_mechanism,
    batch_true,
    truthful_allocation,
    truthful_payment,
    nAgent,
    nObject,
    grid_size=1000,
    grid_chunk_size=100,
):
    B = batch_true.shape[0]
    grid_values = MINIMUM + (MAXIMUM - MINIMUM) * torch.arange(grid_size, device=device) / grid_size
    truthful_u = utility(batch_true, truthful_allocation, truthful_payment)

    max_u = torch.zeros(B, nAgent, nObject, device=device)
    best_misreport_values = torch.zeros(B, nAgent, nObject, device=device)

    with torch.no_grad():
        for agent_idx in range(nAgent):
            for obj_idx in range(nObject):
                best_util = torch.full((B,), -float("inf"), device=device)
                best_value = torch.zeros(B, device=device)

                for start in range(0, grid_size, grid_chunk_size):
                    end = min(start + grid_chunk_size, grid_size)
                    values_chunk = grid_values[start:end]
                    C = values_chunk.shape[0]

                    batch_mis = batch_true.unsqueeze(1).repeat(1, C, 1, 1)
                    batch_mis[:, :, agent_idx, obj_idx] = values_chunk.view(1, C)

                    alloc_m, pay_m = eval_mechanism(batch_mis.reshape(-1, nAgent, nObject))
                    util_grid = (
                        alloc_m.reshape(B, C, nAgent, nObject) * batch_true.unsqueeze(1)
                    ).sum(dim=3) - pay_m.reshape(B, C, nAgent)

                    chunk_best_util, chunk_best_idx = torch.max(util_grid[:, :, agent_idx], dim=1)
                    update_mask = chunk_best_util > best_util
                    best_util[update_mask] = chunk_best_util[update_mask]
                    best_value[update_mask] = values_chunk[chunk_best_idx[update_mask]]

                max_u[:, agent_idx, obj_idx] = best_util - truthful_u[:, agent_idx]
                best_misreport_values[:, agent_idx, obj_idx] = best_value

    return max_u, best_misreport_values


def create_initialization_pools(batch_true, full_best, max_k=80, seed=None, noise_scale_truth=0.2, noise_scale_best=0.2):
    """Generate max_k optional initialization pools once, then slice them for K=1 and K=80."""
    if seed is not None:
        set_seed(seed)

    B, A, M = batch_true.shape
    return {
        "global_random": torch.rand(max_k, B, A, M, device=device),
        "truth_noise": torch.randn(max_k, B, A, M, device=device) * noise_scale_truth,
        "best_noise": torch.randn(max_k, B, A, M, device=device) * noise_scale_best,
    }


def build_misreport_initializations(batch_true, full_best, max_u, k, pools):
    B, A, M = batch_true.shape
    mis_list = []

    # Base candidates from the grid-search stage. These are identical for all K on the same data.
    mis_list.append(full_best)

    for obj_idx in range(M):
        mis_i = batch_true.clone()
        mis_i[:, :, obj_idx] = full_best[:, :, obj_idx]
        mis_list.append(mis_i)

    best_item_idx = torch.argmax(max_u, dim=2)
    mis_bestitem = batch_true.clone()
    b_idx = torch.arange(B, device=device)
    for agent_idx in range(A):
        idx_l = best_item_idx[:, agent_idx]
        mis_bestitem[b_idx, agent_idx, idx_l] = full_best[b_idx, agent_idx, idx_l]
    mis_list.append(mis_bestitem)

    # Optional global random misreport initializations.
    for idx in range(k):
        mis_list.append(pools["global_random"][idx])

    # Optional local perturbations around truthful bids.
    for idx in range(k):
        mis_loc = torch.clamp(batch_true + pools["truth_noise"][idx], 0.0, 1.0)
        mis_list.append(mis_loc)

    # Optional local perturbations around the best misreports.
    for idx in range(k):
        mis_loc_best = torch.clamp(full_best + pools["best_noise"][idx], 0.0, 1.0)
        mis_list.append(mis_loc_best)

    return torch.stack(mis_list, dim=1).detach()


def optimize_misreports(eval_mechanism, batch_true, batch_misreports, R=300, gamma=0.001, init_chunk_size=32):
    batch_misreports.requires_grad_(True)
    opt = Adam([batch_misreports], lr=gamma)
    B, K_total, A, _ = batch_misreports.shape
    denom = float(B * K_total * A)

    for step in range(R):
        opt.zero_grad()

        if init_chunk_size is None or init_chunk_size >= K_total:
            adv_u = misreportUtility(eval_mechanism, batch_true, batch_misreports)
            los = -torch.mean(adv_u).to(device)
            los.backward()
        else:
            for start in range(0, K_total, init_chunk_size):
                end = min(start + init_chunk_size, K_total)
                adv_u_chunk = misreportUtility(eval_mechanism, batch_true, batch_misreports[:, start:end].contiguous())
                los = -torch.sum(adv_u_chunk).to(device) / denom
                los.backward()

        opt.step(restricted=True, min=MINIMUM, max=MAXIMUM)

        if (step + 1) % 50 == 0 or step + 1 == R:
            print(f"  optimization step {step + 1}/{R}")

    eval_mechanism.zero_grad()
    return batch_misreports.detach()


def max_adv_utility(eval_mechanism, batch_true, batch_misreports, init_chunk_size=32):
    K_total = batch_misreports.shape[1]
    chunks = []
    with torch.no_grad():
        if init_chunk_size is None or init_chunk_size >= K_total:
            chunks.append(misreportUtility(eval_mechanism, batch_true, batch_misreports))
        else:
            for start in range(0, K_total, init_chunk_size):
                end = min(start + init_chunk_size, K_total)
                chunks.append(misreportUtility(eval_mechanism, batch_true, batch_misreports[:, start:end].contiguous()))
    return torch.cat(chunks, dim=1).max(dim=1)[0]


def evaluate_one_k_on_fixed_data(eval_mechanism, batch_true, full_best, max_u, pools, nAgent, nObject, k):
    started = time.time()
    batch_misreports = build_misreport_initializations(batch_true, full_best, max_u, k=k, pools=pools)
    candidate_count = batch_misreports.shape[1]
    print(f"  K={k}: candidate initializations {candidate_count} = {nObject + 2} base + 3 * {k}")

    batch_misreports = optimize_misreports(
        eval_mechanism,
        batch_true,
        batch_misreports,
        R=R,
        gamma=GAMMA,
        init_chunk_size=INIT_CHUNK_SIZE,
    )

    with torch.no_grad():
        mis_report_utility_max = max_adv_utility(
            eval_mechanism,
            batch_true,
            batch_misreports,
            init_chunk_size=INIT_CHUNK_SIZE,
        )
        allocation, payment = eval_mechanism(batch_true)
        truthful_utility = utility(batch_true, allocation, payment)
        regret = F.relu(mis_report_utility_max - truthful_utility)
        total_regret = torch.sum(torch.mean(regret, dim=0)).to(device)
        los, r_mean, pay_mean = loss(payment, regret)

    total_regret = float(total_regret.detach().cpu().item())
    payment_value = float(pay_mean.detach().cpu().item())
    opt_revenue = float((-los).detach().cpu().item()) ** 2

    return {
        "K": int(k),
        "total_regret": total_regret,
        "avg_regret_per_bidder": total_regret / nAgent,
        "payment": payment_value,
        "opt_revenue": opt_revenue,
        "candidate_count": int(candidate_count),
        "nBatch": int(N_BATCH),
        "R": int(R),
        "runtime_sec": time.time() - started,
    }


def compare_k_values_on_same_data(model, nAgent, nObject, seed=None):
    if seed is not None:
        set_seed(seed)

    eval_mechanism = get_eval_mechanism(model, use_base_model=USE_BASE_MODEL)
    grad_states = freeze_parameters(eval_mechanism)

    try:
        true = np.random.rand(N_BATCH, nAgent, nObject).astype("float32")
        batch_true = torch.tensor(true, dtype=torch.float32, device=device)

        with torch.no_grad():
            allocation, payment = eval_mechanism(batch_true)

        print("  fixed valuation profiles:", tuple(batch_true.shape))
        print("  grid search for shared best single-item misreports")
        max_u, full_best = grid_best_single_item_misreports(
            eval_mechanism,
            batch_true,
            allocation,
            payment,
            nAgent,
            nObject,
            grid_size=GRID_SIZE,
            grid_chunk_size=GRID_CHUNK_SIZE,
        )

        init_seed = None if seed is None else seed + 100000
        pools = create_initialization_pools(batch_true, full_best, max_k=MAX_K, seed=init_seed)

        rows = []
        for k in K_VALUES:
            row = evaluate_one_k_on_fixed_data(
                eval_mechanism,
                batch_true,
                full_best,
                max_u,
                pools,
                nAgent,
                nObject,
                k=k,
            )
            row["same_data_seed"] = seed
            row["init_pool_seed"] = init_seed
            rows.append(row)

    finally:
        restore_parameters(grad_states)

    return rows


In [4]:
def fmt_cell(value, digits=6):
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def print_table(rows, columns, digits=6):
    if not rows:
        print("(no rows)")
        return
    rendered = [[fmt_cell(row.get(col, ""), digits=digits) for col in columns] for row in rows]
    widths = [max(len(col), *(len(row[idx]) for row in rendered)) for idx, col in enumerate(columns)]
    print(" | ".join(col.ljust(widths[idx]) for idx, col in enumerate(columns)))
    print("-+-".join("-" * width for width in widths))
    for row in rendered:
        print(" | ".join(row[idx].rjust(widths[idx]) for idx in range(len(columns))))


results = []
comparisons = []

for spec in MODEL_SPECS:
    print(f"\n=== {spec['name']} | {spec['path']} ===")
    model = torch_load_full_model(spec["path"])

    for repeat in range(N_REPEATS):
        seed = SEED + repeat
        rows = compare_k_values_on_same_data(
            model,
            nAgent=spec["nAgent"],
            nObject=spec["nObject"],
            seed=seed,
        )

        for row in rows:
            row.update({
                "model": spec["name"],
                "path": str(spec["path"]),
                "repeat": repeat,
            })
            results.append(row)
            print(
                f"  K={row['K']}: regret={row['total_regret']:.5f}  "
                f"avg/bidder={row['avg_regret_per_bidder']:.5f}  "
                f"optRev={row['opt_revenue']:.3f}  "
                f"payment={row['payment']:.3f}  "
                f"time={row['runtime_sec']:.1f}s"
            )

        by_k = {row["K"]: row for row in rows}
        if 1 in by_k and 80 in by_k:
            k1 = by_k[1]
            k80 = by_k[80]
            regret_delta = k80["total_regret"] - k1["total_regret"]
            avg_delta = k80["avg_regret_per_bidder"] - k1["avg_regret_per_bidder"]
            payment_delta = k80["payment"] - k1["payment"]
            regret_delta_pct = regret_delta / max(abs(k1["total_regret"]), 1e-12) * 100.0
            comparisons.append({
                "model": spec["name"],
                "repeat": repeat,
                "same_data_seed": seed,
                "regret_K1": k1["total_regret"],
                "regret_K80": k80["total_regret"],
                "delta_K80_minus_K1": regret_delta,
                "delta_pct_vs_K1": regret_delta_pct,
                "avg_bidder_delta": avg_delta,
                "payment_delta": payment_delta,
                "candidate_K1": k1["candidate_count"],
                "candidate_K80": k80["candidate_count"],
            })
            print(
                f"  Difference on same data: regret(K=80)-regret(K=1)="
                f"{regret_delta:.6f} ({regret_delta_pct:.2f}% vs K=1)"
            )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDetailed results: same valuation profiles within each model/repeat")
print_table(
    results,
    [
        "model", "repeat", "same_data_seed", "init_pool_seed", "K", "candidate_count",
        "total_regret", "avg_regret_per_bidder", "payment", "opt_revenue", "runtime_sec",
    ],
)

print("\nK=80 minus K=1 comparison")
print_table(
    comparisons,
    [
        "model", "repeat", "same_data_seed", "candidate_K1", "candidate_K80",
        "regret_K1", "regret_K80", "delta_K80_minus_K1", "delta_pct_vs_K1",
        "avg_bidder_delta", "payment_delta",
    ],
)



=== 2x2 | 22stage3.pth ===
  fixed valuation profiles: (1000, 2, 2)
  grid search for shared best single-item misreports
  K=1: candidate initializations 7 = 4 base + 3 * 1


/home/wkw/ysy/M2L codes/Stage III/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


  optimization step 50/300
  optimization step 100/300
  optimization step 150/300
  optimization step 200/300
  optimization step 250/300
  optimization step 300/300
  K=80: candidate initializations 244 = 4 base + 3 * 80
  optimization step 50/300
  optimization step 100/300
  optimization step 150/300
  optimization step 200/300
  optimization step 250/300
  optimization step 300/300
  K=1: regret=0.00122  avg/bidder=0.00061  optRev=0.832  payment=0.899  time=5.5s
  K=80: regret=0.00122  avg/bidder=0.00061  optRev=0.832  payment=0.899  time=150.7s
  Difference on same data: regret(K=80)-regret(K=1)=0.000001 (0.06% vs K=1)

=== 2x5 | 25stage3.pt ===
  fixed valuation profiles: (1000, 2, 5)
  grid search for shared best single-item misreports
  K=1: candidate initializations 10 = 7 base + 3 * 1
  optimization step 50/300
  optimization step 100/300
  optimization step 150/300
  optimization step 200/300
  optimization step 250/300
  optimization step 300/300
  K=80: candidate initiali